# 📕 Bài 3 — Pipeline, Leakage & Cross-validation

**Tuần 11**

> Bài này dạy cách viết code ML **đúng chuẩn công nghiệp**. Nội dung ở đây là thứ phân biệt một notebook nghịch chơi với một hệ thống chạy thật.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

MAU = ["#4269d0", "#efb118", "#3ca951", "#ff725c", "#a463f2"]
MAU_CHINH, MAU_NHAN = MAU[0], MAU[3]

DATA = Path("../../../data")
if not (DATA / "churn.csv").exists():
    DATA = Path("data")

df = pd.read_csv(DATA / "churn.csv")
y = df["churn"]
X = df.drop(columns=["churn", "khach_id"])

cot_so = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
cot_chu = [c for c in X.columns if c not in cot_so]

print("Cột số :", cot_so)
print("Cột chữ:", cot_chu)

## 1. 🚨 Leakage do tiền xử lý — nhìn tận mắt

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Dữ liệu nhỏ, nhiều đặc trưng -> leakage lộ rõ nhất
rng = np.random.default_rng(0)
n, p = 60, 300
X_nhieu = pd.DataFrame(rng.normal(size=(n, p)))
y_nhieu = pd.Series(rng.integers(0, 2, n))
# ↑ X hoàn toàn NGẪU NHIÊN, không liên quan gì tới y. AUC thật phải ≈ 0.5

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

cv = StratifiedKFold(5, shuffle=True, random_state=42)

# ❌ SAI: chọn đặc trưng trên TOÀN BỘ dữ liệu rồi mới cross-validate
X_chon = SelectKBest(f_classif, k=10).fit_transform(X_nhieu, y_nhieu)
diem_sai = cross_val_score(LogisticRegression(max_iter=1000), X_chon, y_nhieu,
                           cv=cv, scoring="roc_auc")

# ✅ ĐÚNG: đưa việc chọn đặc trưng VÀO pipeline
pipe_dung = Pipeline([("chon", SelectKBest(f_classif, k=10)),
                      ("model", LogisticRegression(max_iter=1000))])
diem_dung = cross_val_score(pipe_dung, X_nhieu, y_nhieu, cv=cv, scoring="roc_auc")

print(f"  ❌ Chọn đặc trưng TRƯỚC CV : AUC = {diem_sai.mean():.3f}")
print(f"  ✅ Chọn đặc trưng TRONG CV : AUC = {diem_dung.mean():.3f}")
print(f"     Sự thật                 : AUC ≈ 0.500  (X hoàn toàn ngẫu nhiên!)")

> 🚨 **Dữ liệu hoàn toàn ngẫu nhiên, nhưng cách làm sai cho AUC cao chót vót.**
>
> Vì sao? `SelectKBest` nhìn thấy **toàn bộ** `y` để chọn 10 cột "liên quan nhất". Trên 300 cột ngẫu nhiên, chắc chắn có vài cột **tình cờ** tương quan với `y`. Khi cross-validate sau đó, mỗi fold "test" đã góp phần vào việc chọn cột — nó không còn là dữ liệu chưa thấy nữa.
>
> **Đây là loại bug không báo lỗi, chỉ cho bạn một con số đẹp giả tạo.** Rất nhiều bài báo khoa học đã bị rút lại vì chính lỗi này.

## 2. Pipeline — cách sklearn chống leakage cho bạn

In [ ]:
tien_xu_ly = ColumnTransformer([
    ("so", Pipeline([
        ("dien", SimpleImputer(strategy="median")),
        ("chuan_hoa", StandardScaler()),
    ]), cot_so),
    ("chu", OneHotEncoder(handle_unknown="ignore"), cot_chu),
])

model = Pipeline([
    ("tien_xu_ly", tien_xu_ly),
    ("model", LogisticRegression(max_iter=1000)),
])

model

Khi gọi `model.fit(X_train, y_train)`:
- imputer học median **chỉ từ** X_train
- scaler học mean/std **chỉ từ** X_train
- encoder học danh sách hạng mục **chỉ từ** X_train

Khi gọi `model.predict(X_test)`: mọi bước chỉ `transform`, **không fit lại**.

Quan trọng hơn: với cross-validation, **mỗi fold tự tiền xử lý riêng**. Bạn không thể vô tình rò rỉ được nữa.

In [ ]:
# Pipeline nhận DataFrame THÔ — không cần tiền xử lý tay
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

model.fit(X_train, y_train)
print("Dự đoán trên dữ liệu thô:", model.predict(X_test.head(5)))
print("Xác suất:", model.predict_proba(X_test.head(5))[:, 1].round(3))

In [ ]:
# Xử lý được cả hạng mục CHƯA TỪNG THẤY (nhờ handle_unknown="ignore")
X_la = X_test.head(3).copy()
X_la["loai_hop_dong"] = "GOI_MOI_CHUA_TUNG_CO"
print("Không sập:", model.predict(X_la))

> 📌 Trong sản xuất, tháng sau xuất hiện một loại hợp đồng mới. Mặc định sklearn **ném lỗi** và API của bạn trả về 500. Với `handle_unknown="ignore"`, model vẫn dự đoán được (kém chính xác hơn, nhưng hệ thống không sập).

## 3. Cross-validation

In [ ]:
diem = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc")

print("Điểm từng fold:", diem.round(4))
print(f"\n  AUC = {diem.mean():.4f} ± {diem.std():.4f}")
print(f"  thấp nhất {diem.min():.4f} — cao nhất {diem.max():.4f}")

```
Fold 1: [TEST][train][train][train][train]
Fold 2: [train][TEST][train][train][train]
Fold 3: [train][train][TEST][train][train]
Fold 4: [train][train][train][TEST][train]
Fold 5: [train][train][train][train][TEST]
                                          → 5 điểm → mean ± std
```

> 💡 **Độ lệch chuẩn quan trọng ngang trung bình.**
> `0.85 ± 0.02` → model ổn định, tin được.
> `0.85 ± 0.12` → điểm số phần lớn do may rủi, **không** tin được.
>
> Báo cáo chỉ có mean là báo cáo thiếu một nửa sự thật.

**`StratifiedKFold`** giữ tỷ lệ lớp trong mỗi fold — luôn dùng cho phân loại.

> ⚠️ **Chuỗi thời gian:** tuyệt đối không dùng KFold thường (xáo trộn khiến model train trên tương lai, test trên quá khứ). Dùng `TimeSeriesSplit`.

## 4. ⭐⭐ Săn đặc trưng rò rỉ

In [ ]:
def auc_mot_cot(cot):
    Xc = X[[cot]]
    cs = [cot] if pd.api.types.is_numeric_dtype(X[cot]) else []
    cc = [] if cs else [cot]
    pre = ColumnTransformer([
        ("so", Pipeline([("d", SimpleImputer(strategy="median")),
                         ("s", StandardScaler())]), cs),
        ("chu", OneHotEncoder(handle_unknown="ignore"), cc),
    ])
    pipe = Pipeline([("pre", pre), ("m", LogisticRegression(max_iter=1000))])
    return cross_val_score(pipe, Xc, y, cv=StratifiedKFold(3, shuffle=True, random_state=42),
                           scoring="roc_auc").mean()


diem_le = pd.Series({c: auc_mot_cot(c) for c in X.columns}).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
mau_cot = [MAU_NHAN if v >= 0.9 else MAU_CHINH for v in diem_le]
ax.barh(diem_le.index[::-1], diem_le.to_numpy()[::-1], color=mau_cot[::-1], height=.65)
ax.axvline(0.5, color="gray", ls=":", label="đoán mò")
ax.axvline(0.9, color=MAU_NHAN, ls="--", label="ngưỡng nghi ngờ")
ax.set_xlabel("AUC khi dùng MỘT MÌNH cột này")
ax.set_title("Săn đặc trưng rò rỉ")
ax.grid(axis="y", visible=False); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

print(diem_le.round(3).to_string())

Cột `da_goi_tong_dai_huy` một mình đạt AUC ≈ 0.92 — **cao hơn tất cả các cột khác cộng lại**. Đó là cờ đỏ.

Giờ đặt câu hỏi quyết định:

> ### *"Lúc tôi CẦN dự đoán, tôi đã có cột này chưa?"*

**Chưa.** Khách gọi tổng đài để huỷ **sau khi** đã quyết định rời bỏ. Đưa nó vào model là hỏi *"khách sắp huỷ không?"* bằng thông tin *"khách đã gọi huỷ rồi"*.

In [ ]:
def auc_cv(cols):
    Xc = X[cols]
    cs = [c for c in cols if pd.api.types.is_numeric_dtype(X[c])]
    cc = [c for c in cols if c not in cs]
    pre = ColumnTransformer([
        ("so", Pipeline([("d", SimpleImputer(strategy="median")),
                         ("s", StandardScaler())]), cs),
        ("chu", OneHotEncoder(handle_unknown="ignore"), cc),
    ])
    pipe = Pipeline([("pre", pre), ("m", LogisticRegression(max_iter=1000))])
    return cross_val_score(pipe, Xc, y, cv=cv, scoring="roc_auc").mean()


tat_ca = list(X.columns)
sach = [c for c in tat_ca if c != "da_goi_tong_dai_huy"]

print(f"  CÓ cột rò rỉ    : AUC = {auc_cv(tat_ca):.4f}   ← đẹp giả tạo")
print(f"  BỎ cột rò rỉ    : AUC = {auc_cv(sach):.4f}   ← con số THẬT")
print()
print("  Nếu triển khai model đầu tiên, ngoài đời nó sẽ hoạt động")
print("  ở mức con số thứ hai — và bạn sẽ không hiểu vì sao.")

### Ba dạng leakage

| Dạng | Ví dụ | Phát hiện bằng |
|---|---|---|
| **Tiền xử lý** | `fit` scaler/selector trên toàn bộ dữ liệu | Dùng Pipeline là hết |
| **Đặc trưng** | Cột chỉ tồn tại *sau* sự kiện cần dự đoán | Hỏi *"lúc dự đoán tôi có cột này chưa?"* |
| **Thời gian** | Train trên tháng 12, test trên tháng 6 | Chia theo mốc thời gian |

### Ba câu hỏi phát hiện leakage

```
① Kết quả có tốt bất thường không?         AUC > 0.95 ở bài toán khó → nghi ngờ
② Một đặc trưng chiếm gần hết importance?   → nghi ngờ
③ Lúc dự đoán THẬT, tôi có cột này chưa?    → câu hỏi quyết định
```

> 🎯 **Nguyên tắc vàng: kết quả đẹp bất ngờ là tin XẤU cho tới khi chứng minh được ngược lại.**

## 5. Đa cộng tuyến — cái bẫy thứ hai

In [ ]:
tq = X[["so_thang_su_dung", "cuoc_hang_thang", "tong_chi_tieu"]].corr()

fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(tq, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(3), tq.columns, rotation=30, ha="right")
ax.set_yticks(range(3), tq.columns)
ax.grid(visible=False)
for i in range(3):
    for j in range(3):
        v = tq.iloc[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                color="white" if abs(v) > .55 else "#222")
plt.colorbar(im, ax=ax, shrink=.8)
plt.title("Tương quan giữa các đặc trưng")
plt.tight_layout(); plt.show()

`tong_chi_tieu ≈ so_thang_su_dung × cuoc_hang_thang` → tương quan rất cao.

**Hậu quả với model tuyến tính:**
- Hệ số trở nên **không ổn định** — đổi vài dòng dữ liệu, hệ số nhảy loạn, thậm chí đổi dấu
- Không thể diễn giải: *"mỗi tháng sử dụng thêm làm giảm 3% khả năng rời bỏ"* không còn đúng vì hai biến đi cùng nhau
- Điểm dự đoán thường **không** bị ảnh hưởng nhiều

**Xử lý:** bỏ bớt một biến, hoặc gộp thành biến mới (`chi_tieu_trung_binh = tong / so_thang`), hoặc dùng Ridge.

> Model dạng **cây** không bị vấn đề này — chúng chỉ chọn một trong hai biến để chia.

## 6. Tổng kết

```python
# Khung code ML đúng chuẩn — dùng lại cho mọi dự án
pre = ColumnTransformer([
    ("so",  Pipeline([("dien", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), cot_so),
    ("chu", OneHotEncoder(handle_unknown="ignore"), cot_chu),
])
pipe = Pipeline([("pre", pre), ("model", LogisticRegression(max_iter=1000))])

cv = StratifiedKFold(5, shuffle=True, random_state=42)
diem = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")
print(f"AUC = {diem.mean():.4f} ± {diem.std():.4f}")
```

**Ba thói quen bắt buộc:**
1. **Mọi** tiền xử lý nằm trong Pipeline — không bao giờ làm tay trước CV
2. Săn đặc trưng rò rỉ **trước** khi ăn mừng kết quả
3. Báo cáo mean **và** std

## ✍️ Bài tập

```bash
pytest tests/phase04/test_ex03.py -v
```